In [36]:
!pip install ultralytics opencv-python torch torchvision -q

In [37]:
import cv2
import os
import glob
import torch
from ultralytics import YOLO
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [38]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
video_folder = "/content/drive/MyDrive/videos_dataset"
video_files = glob.glob(video_folder + "/*.mp4")

print("Found videos:")
for v in video_files:
    print(v)

Found videos:
/content/drive/MyDrive/videos_dataset/video_4.mp4
/content/drive/MyDrive/videos_dataset/video_7.mp4
/content/drive/MyDrive/videos_dataset/video_2.mp4
/content/drive/MyDrive/videos_dataset/video_8.mp4
/content/drive/MyDrive/videos_dataset/video_1.mp4
/content/drive/MyDrive/videos_dataset/video_3.mp4
/content/drive/MyDrive/videos_dataset/video_6.mp4
/content/drive/MyDrive/videos_dataset/video_5.mp4


In [40]:
# Player detection + tracking model
model = YOLO("yolov8n.pt")

# Pose estimation model (keypoints)
pose_model = YOLO("yolov8n-pose.pt")


In [41]:
output_folder = "/content/drive/MyDrive/videos_output"
os.makedirs(output_folder, exist_ok=True)

print("Output folder:", output_folder)

Output folder: /content/drive/MyDrive/videos_output


In [33]:
for i, video_path in enumerate(video_files, start=1):
    print(f"\nProcessing TRACKING video {i}: {video_path}")

    cap = cv2.VideoCapture(video_path)

    # Get video properties (IMPORTANT FIX)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = os.path.join(output_folder, f"tracked_video_{i}.mp4")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # =========================
        # YOLOv8 TRACKING (KEY FIX)
        # =========================
        results = model.track(
          frame,
          persist=True,
          classes=[0],
          tracker="bytetrack.yaml",
          verbose=False
)

        annotated_frame = results[0].plot()
        out.write(annotated_frame)

    cap.release()
    out.release()

    print(f"✅ Tracking saved: {output_path}")


Processing TRACKING video 1: /content/drive/MyDrive/videos_dataset/video_4.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_1.mp4

Processing TRACKING video 2: /content/drive/MyDrive/videos_dataset/video_7.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_2.mp4

Processing TRACKING video 3: /content/drive/MyDrive/videos_dataset/video_2.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_3.mp4

Processing TRACKING video 4: /content/drive/MyDrive/videos_dataset/video_8.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_4.mp4

Processing TRACKING video 5: /content/drive/MyDrive/videos_dataset/video_1.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_5.mp4

Processing TRACKING video 6: /content/drive/MyDrive/videos_dataset/video_3.mp4
✅ Tracking saved: /content/drive/MyDrive/videos_output/tracked_video_6.mp4

Processing TRACKING video 7: /content/drive/MyDrive/videos_dataset/vi

In [34]:
for i, video_path in enumerate(video_files, start=1):
    print(f"\nProcessing POSE video {i}: {video_path}")

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = os.path.join(output_folder, f"pose_video_{i}.mp4")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Pose estimation
        results = pose_model(frame)

        annotated_frame = results[0].plot()
        out.write(annotated_frame)

    cap.release()
    out.release()

    print(f"🎯 Pose saved: {output_path}")

Streaming output truncated to the last 5000 lines.
Speed: 3.5ms preprocess, 21.1ms inference, 4.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 19.3ms
Speed: 7.0ms preprocess, 19.3ms inference, 4.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 20.6ms
Speed: 6.8ms preprocess, 20.6ms inference, 4.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 persons, 19.8ms
Speed: 3.6ms preprocess, 19.8ms inference, 4.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 persons, 15.3ms
Speed: 5.0ms preprocess, 15.3ms inference, 3.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 persons, 15.4ms
Speed: 5.2ms preprocess, 15.4ms inference, 3.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 persons, 14.1ms
Speed: 4.3ms preprocess, 14.1ms inference, 4.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 persons, 18.0ms
Speed: 6.0ms preprocess, 18.0ms inference, 3.8ms postprocess p

In [35]:
print("\n✅ ALL PROCESSING COMPLETE")
print("Output folder:", output_folder)

!ls -lh "$output_folder"


✅ ALL PROCESSING COMPLETE
Output folder: /content/drive/MyDrive/videos_output
total 274M
-rw------- 1 root root 8.0M Jun 30 10:24 pose_video_1.mp4
-rw------- 1 root root  30M Jun 30 10:25 pose_video_2.mp4
-rw------- 1 root root  17M Jun 30 10:25 pose_video_3.mp4
-rw------- 1 root root 6.2M Jun 30 10:25 pose_video_4.mp4
-rw------- 1 root root  52M Jun 30 10:26 pose_video_5.mp4
-rw------- 1 root root  17M Jun 30 10:26 pose_video_6.mp4
-rw------- 1 root root 8.8M Jun 30 10:27 pose_video_7.mp4
-rw------- 1 root root 8.5M Jun 30 10:27 pose_video_8.mp4
-rw------- 1 root root 5.5M Jun 30 10:23 tracked_video_1.mp4
-rw------- 1 root root  27M Jun 30 10:23 tracked_video_2.mp4
-rw------- 1 root root  16M Jun 30 10:23 tracked_video_3.mp4
-rw------- 1 root root 9.2M Jun 30 10:23 tracked_video_4.mp4
-rw------- 1 root root  42M Jun 30 10:24 tracked_video_5.mp4
-rw------- 1 root root  13M Jun 30 10:24 tracked_video_6.mp4
-rw------- 1 root root 9.6M Jun 30 10:24 tracked_video_7.mp4
-rw------- 1 root r